# Qwen3 한국어 보이스 클론 — Quickstart

본인 음성 한 문장(약 5~15초) 샘플로 한국어 TTS를 클로닝합니다.

**실행 전 준비**
- 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 → 저장
- 본인 음성 wav 파일 1개 준비 (한 문장(약 5~15초), 조용한 환경에서 대화 톤으로 녹음)

**실행 방법**: 셀을 순서대로 실행하거나 메뉴 → 런타임 → 모두 실행.

리포: https://github.com/azzselloo-sudo/qwen3-korean-voice-clone

## 1. 패키지 설치

In [1]:
!pip install -q qwen-tts soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.5 MB/s eta 0:00:00


## 2. 참고 음성 업로드

셀 실행 → 파일 선택 → 본인 한 문장(약 5~15초) 녹음 wav 업로드.

In [2]:
from google.colab import files
print('본인 녹음 wav 파일을 업로드하세요 (한 문장(약 5~15초))')
up = files.upload()
ref_filename = list(up.keys())[0]
print(f'업로드 완료: {ref_filename}')


본인 녹음 wav 파일을 업로드하세요 (한 문장(약 5~15초))


Saving female_teenager_pitch0.wav to female_teenager_pitch0.wav
업로드 완료: female_teenager_pitch0.wav


## 3. Qwen3-TTS 모델 로드

첫 실행 시 약 3.5GB 가중치 다운로드 (~3분). 동일 세션 내 재실행은 캐시.

In [3]:
import torch
from qwen_tts import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained(
    'Qwen/Qwen3-TTS-12Hz-1.7B-Base',
    device_map='cuda:0',
    dtype=torch.float16,
    attn_implementation='eager',
)
print('모델 로딩 완료')



    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

모델 로딩 완료


## 4. 텍스트 입력

- `REF_TEXT`: 업로드한 음성에서 실제로 발화한 텍스트(정확히 입력)
- `TEXT`: 본인 목소리로 생성할 문장

In [4]:
# 본인 녹음에서 실제로 발화한 텍스트
REF_TEXT = '안녕하세요. 오늘은 별자리 이야기를 들려줄게요. 밤하늘을 함께 올려다봐요.'

# 생성할 문장 (~120자 권장)
TEXT = '모아 온 쓰레기를 분류한 뒤 카메라로 촬영하여 이미지 데이터로 만들어 볼까요?'


## 5. 음성 생성

In [5]:
import numpy as np, soundfile as sf
from IPython.display import Audio, display

wavs, sr = model.generate_voice_clone(
    text=TEXT,
    language='Korean',
    ref_audio=ref_filename,
    ref_text=REF_TEXT,
)
if isinstance(wavs, torch.Tensor):
    wavs = wavs.detach().cpu().float().numpy()
wavs = np.asarray(wavs, dtype=np.float32)
if wavs.ndim > 1:
    wavs = wavs[0]

sf.write('output.wav', wavs, int(sr))
print(f'생성 완료: {len(wavs)/sr:.1f}초')
display(Audio('output.wav'))


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


생성 완료: 5.4초


## 6. 결과 다운로드 (선택)

In [ ]:
from google.colab import files
files.download('output.wav')


---

## 운영 팁

- 참고 음성의 품질이 결과 품질을 결정합니다. 조용한 환경에서 대화 톤으로 녹음하세요.
- `REF_TEXT`는 녹음 내용과 일치할수록 결과 정확도가 높습니다.
- 한 번에 긴 문장을 생성하려면 셀 4의 `TEXT`를 늘리되 ~120자 단위로 분할하는 편이 자연스럽습니다.
- Colab 무료 티어는 일 약 12시간. 대량 처리는 Colab Pro 또는 Modal 등 서버리스 환경으로 이전.

## 자주 발생하는 오류

| 증상 | 해결 |
|---|---|
| `No module named 'qwen_tts'` | 셀 1 재실행 |
| `CUDA out of memory` | 런타임 → 세션 다시 시작 → 처음부터 재실행 |
| 발음 부자연스러움 | 참고 음성 재녹음 (조용한 환경, 대화 톤) |
| `files.upload()` 무반응 | 브라우저 새로고침 후 셀 2 재실행 |

리포: https://github.com/azzselloo-sudo/qwen3-korean-voice-clone